<h1>Chapter 9 - Graph RAG: Recipe 5: Useful Extensions</h1>
<i>Building knowledge graphs and using them in RAG systems.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch09_graph_rag/9.5_useful_extensions.ipynb)

---

This notebook is for Chapter 9 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


## 0. Install Required Packages

First, install all the necessary packages for this notebook.

In [ ]:
%pip install python-dotenv neo4j openai

## 1. Setup and Imports

Import required libraries and load environment variables.

In [ ]:
from __future__ import annotations
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase
from openai import OpenAI

# Load environment variables from .env file
load_dotenv()

## 2. Connect to Neo4j and OpenAI

Create a reusable driver function and initialize the OpenAI client.

Make sure your `.env` file contains:
```
NEO4J_URI=neo4j://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your_password
OPENAI_API_KEY=your_openai_api_key
```

In [ ]:
def get_driver():
    """Create Neo4j driver instance from environment variables."""
    uri = os.getenv("NEO4J_URI", "neo4j://localhost:7687")
    user = os.getenv("NEO4J_USERNAME", "neo4j")
    pwd = os.getenv("NEO4J_PASSWORD")
    return GraphDatabase.driver(uri, auth=(user, pwd))

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Test connection
driver = get_driver()
print("✓ Connected to Neo4j")
driver.close()

## 3. Helper Function: Create Embeddings

Define a reusable function to generate embeddings for text.

In [ ]:
def create_embedding(text: str) -> list[float]:
    """Generate embeddings using OpenAI's text-embedding-3-small model."""
    response = client.embeddings.create(model="text-embedding-3-small", input=text)
    return response.data[0].embedding

print("✓ Embedding function ready")

## 4. Extension 1: Add Clause Summaries

Generate concise summaries for each clause using GPT-4o-mini. This helps with quick comprehension and can improve retrieval by providing alternative representations of the clause content.

In [ ]:
def summarize_clause(text: str):
    """Generate a summary for a clause using GPT-4o-mini."""
    prompt = f"Summarize this SLA clause:\n\n{text}"
    return (
        client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=120,
        )
        .choices[0]
        .message.content
    )

def add_clause_summaries():
    """Add summaries to all Clause nodes that don't have one."""
    driver = get_driver()
    with driver.session() as session:
        result = session.run(
            "MATCH (cl:Clause) WHERE cl.summary IS NULL RETURN cl.id AS id, cl.text AS text"
        )
        clauses = list(result)
        print(f"Generating summaries for {len(clauses)} clauses...")
        
        for i, row in enumerate(clauses, 1):
            s = summarize_clause(row["text"])
            session.run(
                "MATCH (cl:Clause {id:$id}) SET cl.summary=$s", id=row["id"], s=s
            )
            if i % 5 == 0:
                print(f"  Processed {i}/{len(clauses)} clauses")
    driver.close()
    print("✓ Added clause summaries")

# Execute (uncomment to run)
add_clause_summaries()

## 5. Extension 2: Precompute Retrieval Shortcuts

Precompute useful aggregates for faster queries. This trades some storage space for significant query speed improvements.

In [ ]:
def precompute_retrieval_shortcuts():
    """Precompute useful aggregates for faster queries."""
    driver = get_driver()
    with driver.session() as session:
        # Count clauses by type for each company
        session.run(
            """
            MATCH (c:Company)-[:HAS_SLA]->(:SLA)-[:HAS_CLAUSE]->(cl:Clause)-[:OF_TYPE]->(t:ClauseType)
            WITH c, t, count(cl) AS num_clauses
            SET c[t.name + '_count'] = num_clauses
            """
        )
        
        # Verify the precomputed data
        result = session.run(
            """
            MATCH (c:Company)
            WHERE c.Termination_count IS NOT NULL
            RETURN c.name AS company, c.Termination_count AS termination_clauses
            LIMIT 5
            """
        )
        
        companies = list(result)
        print("\nSample precomputed data:")
        for row in companies:
            print(f"  {row['company']}: {row['termination_clauses']} termination clauses")
    
    driver.close()
    print("\n✓ Precomputed retrieval shortcuts")

# Execute (uncomment to run)
precompute_retrieval_shortcuts()